# Introdução prática ao Gemini API — Free Tier

Este notebook mostra o básico para interagir com uma IA usando a **Gemini API** em Python.

Você vai aprender a:

1. instalar a biblioteca oficial `google-genai`;
2. configurar a chave da API;
3. enviar uma pergunta simples para o Gemini;
4. usar instrução de sistema;
5. controlar criatividade com `temperature`;
6. criar um chat simples com histórico;
7. gerar respostas estruturadas em JSON.

> Material pensado para Google Colab e para uso didático em aula.

## 1. Instalação da biblioteca

A biblioteca `google-genai` permite acessar os modelos Gemini usando Python.

No Google Colab, execute a célula abaixo para instalar o pacote.

In [8]:
!pip install google-genai

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuração da chave da API

Para usar a Gemini API, você precisa de uma chave de API.

Ela pode ser criada no Google AI Studio.

Nesta célula, usamos `getpass` para esconder a chave enquanto ela é digitada.

In [ ]:
import os
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Cole sua GEMINI_API_KEY: ")

## 3. Criando o cliente Gemini

O cliente é o objeto responsável por se comunicar com a API do Gemini.

In [4]:
from google import genai

cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

KeyError: 'GEMINI_API_KEY'

## 4. Escolhendo o modelo

Para aulas e testes rápidos no free tier, modelos da família **Flash** costumam ser uma boa escolha.

Caso o modelo abaixo não esteja disponível na sua conta, troque por outro modelo Flash disponível no Google AI Studio.

In [6]:
MODELO = "gemini-2.5-flash"

## 5. Primeira pergunta para a IA

Agora vamos enviar uma pergunta simples para testar se a conexão está funcionando.

In [ ]:
resposta = cliente.models.generate_content(
    model=MODELO,
    contents="Explique em poucas palavras o que é inteligência artificial."
)

print(resposta.text)

## 6. Criando uma função para perguntar ao Gemini

Agora vamos organizar o código em uma função reutilizável.

Assim, sempre que quisermos fazer uma pergunta, basta chamar `perguntar_ao_gemini()`.

In [ ]:
def perguntar_ao_gemini(pergunta: str) -> str:
    """
    Envia uma pergunta para o Gemini e retorna a resposta em texto.
    """

    resposta = cliente.models.generate_content(
        model=MODELO,
        contents=pergunta
    )

    return resposta.text

In [ ]:
resposta = perguntar_ao_gemini("Me dê 3 exemplos de uso de IA no dia a dia.")
print(resposta)

## 7. Usando uma instrução de sistema

A **instrução de sistema** define como a IA deve se comportar.

Exemplo: responder em português, ser objetiva, agir como professora, explicar passo a passo etc.

In [5]:
INSTRUCAO_SISTEMA = (
    "Você é um professor de tecnologia. "
    "Explique os assuntos em português do Brasil, "
    "com linguagem simples, exemplos práticos e sem respostas longas demais."
)

resposta = cliente.models.generate_content(
    model=MODELO,
    contents="Explique o que é uma API para um aluno iniciante.",
    config={
        "system_instruction": INSTRUCAO_SISTEMA
    }
)

print(resposta.text)

NameError: name 'cliente' is not defined

## 8. Controlando a criatividade com `temperature`

O parâmetro `temperature` controla o nível de criatividade da resposta.

- Valores baixos, como `0.1` ou `0.2`, deixam a resposta mais direta e previsível.
- Valores médios, como `0.5`, equilibram clareza e criatividade.
- Valores altos, como `0.9`, deixam a resposta mais criativa, mas menos previsível.

In [ ]:
pergunta = "Crie uma analogia simples para explicar o que é machine learning."

resposta = cliente.models.generate_content(
    model=MODELO,
    contents=pergunta,
    config={
        "temperature": 0.7,
        "system_instruction": INSTRUCAO_SISTEMA
    }
)

print(resposta.text)

## 9. Limitando o tamanho da resposta

O parâmetro `max_output_tokens` ajuda a limitar o tamanho da resposta.

Isso é útil para economizar uso da API e evitar respostas muito longas.

In [ ]:
resposta = cliente.models.generate_content(
    model=MODELO,
    contents="Explique o que é Python em uma resposta curta.",
    config={
        "system_instruction": INSTRUCAO_SISTEMA,
        "max_output_tokens": 120
    }
)

print(resposta.text)

## 10. Chat simples com histórico

Um chatbot precisa lembrar das mensagens anteriores.

Para isso, vamos guardar o histórico em uma lista.

Cada mensagem será armazenada com um papel:

- `user`: mensagem enviada pelo usuário;
- `model`: resposta gerada pela IA.

In [ ]:
historico = []

def conversar(mensagem_usuario: str) -> str:
    """
    Envia uma mensagem para o Gemini usando o histórico da conversa.
    """

    historico.append({
        "role": "user",
        "parts": [{"text": mensagem_usuario}]
    })

    resposta = cliente.models.generate_content(
        model=MODELO,
        contents=historico,
        config={
            "system_instruction": INSTRUCAO_SISTEMA,
            "temperature": 0.4
        }
    )

    texto_resposta = resposta.text

    historico.append({
        "role": "model",
        "parts": [{"text": texto_resposta}]
    })

    return texto_resposta

In [ ]:
print(conversar("Meu nome é Ana e estou aprendendo Python."))

In [ ]:
print(conversar("Qual é o meu nome e o que eu estou aprendendo?"))

## 11. Limpando o histórico

Se quisermos começar uma conversa nova, basta limpar a lista `historico`.

In [ ]:
historico.clear()
print("Histórico limpo!")

## 12. Pedindo resposta em formato JSON

Às vezes queremos que a IA responda em um formato estruturado.

Isso é útil quando queremos usar a resposta dentro de outro programa.

In [7]:
prompt = """
Crie um plano de estudos simples sobre Python básico.

Responda somente em JSON válido, seguindo este formato:
{
  "tema": "",
  "duracao_dias": 0,
  "topicos": [],
  "projeto_final": ""
}
"""

resposta = cliente.models.generate_content(
    model=MODELO,
    contents=prompt,
    config={
        "temperature": 0.2,
        "system_instruction": "Você responde somente em JSON válido, sem texto antes ou depois."
    }
)

print(resposta.text)

NameError: name 'cliente' is not defined

## 13. Exercícios para os alunos

Faça as atividades abaixo:

1. Altere o prompt da primeira pergunta para pedir uma explicação sobre banco de dados.
2. Mude a `temperature` para `0.1` e depois para `0.9`. Compare as respostas.
3. Crie uma instrução de sistema dizendo que a IA deve responder como um professor do SENAI.
4. Use o chat com histórico para fazer três perguntas conectadas sobre Python.
5. Peça para a IA gerar uma lista de exercícios em JSON.

## 14. Desafio final

Crie uma função chamada `gerar_exercicios()` que receba um tema e retorne 5 exercícios sobre esse tema.

Exemplo esperado:

```python
gerar_exercicios("listas em Python")
```

A função deve pedir para o Gemini gerar exercícios com:

- enunciado;
- nível de dificuldade;
- resposta esperada.

In [7]:
print(gerar_exercicios("listas em Python"))

NameError: name 'gerar_exercicios' is not defined

## Fechamento

Neste notebook você viu o básico para usar o Gemini com Python:

- instalação da biblioteca;
- configuração da chave;
- envio de prompts;
- instruções de sistema;
- parâmetros básicos;
- histórico de conversa;
- resposta estruturada;
- criação de exercícios com IA.

A partir daqui, você pode evoluir para projetos com interface web usando Streamlit ou Gradio.

In [3]:
import os
import gradio as gr
from google import genai
from google.genai import types

cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODELO = "gemini-2.5-flash"

SYSTEM_INSTRUCTION = """
Você é um chatbot educado e objetivo.
Responda em Português do Brasil, com clareza e passos práticos.
Se faltar informação, faça 1 pergunta curta antes de responder.
"""


def converter_historico_para_gemini(historico, mensagem_usuario):
    conteudo = []

    if historico is None:
        historico = []

    for item in historico:
        role = item["role"]
        texto = item["content"]

        if role == "user":
            role_gemini = "user"
        elif role == "assistant":
            role_gemini = "model"
        else:
            continue

        conteudo.append(
            types.Content(
                role=role_gemini,
                parts=[types.Part.from_text(text=texto)]
            )
        )

    conteudo.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=mensagem_usuario)]
        )
    )

    return conteudo


def chat_com_historico(mensagem_usuario, historico):
    conteudo = converter_historico_para_gemini(historico, mensagem_usuario)

    resposta = cliente.models.generate_content(
        model=MODELO,
        contents=conteudo,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.4,
        ),
    )

ModuleNotFoundError: No module named 'gradio'

In [2]:
import gradio as gr


ModuleNotFoundError: No module named 'gradio'